# Milestone 2: Exploratory Data Analysis (EDA)
## PM2.5 & PM10 Air Quality Forecasting - Korba (CECB / CPCB Data)

This notebook inspects multi-station air quality and meteorological patterns across Korba:
- 15-minute raw reading distributions & completeness
- Diurnal pollution cycles (twin peak morning/evening phenomenon)
- PM2.5 vs PM10 correlation & ratio distributions
- Meteorological drivers (Wind Speed & Atmospheric Dispersion)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.fetch import load_all_stations
from src.data.audit import audit_station_data, generate_quality_report_markdown

# Load all available stations from data/raw
DATA_DIR = Path("../data/raw")
stations = load_all_stations(DATA_DIR)
print(f"Loaded stations: {list(stations.keys())}")

In [ ]:
# Run Quality Audit
audit_results = [audit_station_data(df, name) for name, df in stations.items()]
print(generate_quality_report_markdown(audit_results))

In [ ]:
# Diurnal Cycle Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, df in stations.items():
    hourly = df.groupby(df.index.hour)[["PM2.5", "PM10"]].mean()
    axes[0].plot(hourly.index, hourly["PM2.5"], marker="o", label=f"{name} (PM2.5)")
    axes[1].plot(hourly.index, hourly["PM10"], marker="s", label=f"{name} (PM10)")

axes[0].set_title("Hourly Diurnal Profile: PM2.5", fontweight="bold")
axes[0].set_xlabel("Hour of Day (IST)")
axes[0].set_ylabel("Mean PM2.5 (ug/m3)")
axes[0].legend()

axes[1].set_title("Hourly Diurnal Profile: PM10", fontweight="bold")
axes[1].set_xlabel("Hour of Day (IST)")
axes[1].set_ylabel("Mean PM10 (ug/m3)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# PM2.5 vs PM10 Scatter & Physical Bound Ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, df in stations.items():
    clean = df[["PM2.5", "PM10"]].dropna()
    if not clean.empty:
        axes[0].scatter(clean["PM10"], clean["PM2.5"], alpha=0.3, s=15, label=name)
        ratio = clean["PM2.5"] / clean["PM10"]
        axes[1].hist(ratio, bins=35, alpha=0.5, density=True, label=name)

axes[0].plot([0, 500], [0, 500], "r--", label="1:1 Equality line")
axes[0].set_title("PM2.5 vs PM10 Concentration Relationship", fontweight="bold")
axes[0].set_xlabel("PM10 (ug/m3)")
axes[0].set_ylabel("PM2.5 (ug/m3)")
axes[0].legend()

axes[1].set_title("PM2.5 / PM10 Ratio Distribution", fontweight="bold")
axes[1].set_xlabel("Ratio (PM2.5 / PM10)")
axes[1].axvline(1.0, color="red", linestyle="--", label="Physical Bound (1.0)")
axes[1].legend()
plt.tight_layout()
plt.show()